In [1]:
import pandas as pd

df = pd.read_csv("train.csv")
print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    str    
 1   Gender             601 non-null    str    
 2   Married            611 non-null    str    
 3   Dependents         599 non-null    str    
 4   Education          614 non-null    str    
 5   Self_Employed      582 non-null    str    
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    str    
 12  Loan_Status        614 non-null    str    
dtypes: float64(4), int64(1), str(8)
memory usage: 62.5 KB
None
    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Graduate  

In [2]:
print(df.isnull().sum())

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64


In [3]:
df = df.drop(columns= ['Loan_ID'])

In [4]:
print(df['Credit_History'].value_counts())

Credit_History
1.0    475
0.0     89
Name: count, dtype: int64


In [5]:
from sklearn.base import TransformerMixin, BaseEstimator # forgot the base estimator

class Binary_Mapper(BaseEstimator, TransformerMixin):
    def fit(self, X, y= None): #wrote small none
        return self
    def transform(self, X): #forgot the arguments/parameters here hehe ;p
        X = X.copy()
        X['Gender'] = X['Gender'].map({'Male':1, 'Female': 0}) # totally forgot how to write this line haha
        X['Married'] = X['Married'].map({'Yes': 1, 'No': 0})
        X['Education'] = X['Education'].map({'Graduate': 1, 'Not Graduate': 0})
        X['Self_Employed'] = X['Self_Employed'].map({'Yes': 1, 'No': 0})
        return X
        

In [6]:
df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y = None):
        self.median_ = X['LoanAmount'].median()
        return self
    def transform(self, X):
        X = X.copy()
        X['Income_to_Loan'] = X['ApplicantIncome']/X['LoanAmount'].fillna(self.median_)
        return X


In [8]:
categorical_columns = ['Dependents', 'Property_Area']
numeric_columns = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Income_to_Loan']
binary_columns = ['Gender', 'Married', 'Education', 'Self_Employed', 'Credit_History']


In [9]:
from sklearn.model_selection import train_test_split # forgot this module too huhu

X = df.drop(columns=['Loan_Status'])
y = df["Loan_Status"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state= 42, stratify= y) # forgot the test size and stratify writing here.. 


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy= 'most_frequent')),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown= 'ignore', drop = 'first'))
])

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy= 'median')),
    ('scaler', StandardScaler())
])

#forgot all the modules and pipeline syntax - like wth is this ;-;

In [11]:
from sklearn.compose import ColumnTransformer   # Forgot about it entirely ;-;

preprocessor = ColumnTransformer([
    ("bin", binary_pipeline, binary_columns), 
    ("cat", categorical_pipeline, categorical_columns),
    ("num", numeric_pipeline, numeric_columns)
])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score # Yeah forgot all these too ;-;

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state= 42, class_weight= 'balanced'),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    "RandomForestClassifier": RandomForestClassifier(random_state=42, class_weight= 'balanced')
}

In [13]:
for name, model in models.items():
    pipe = Pipeline([
        ('Binary_Mapper', Binary_Mapper()),
        ('Feature_Engineer', FeatureEngineer()),
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv = 5, scoring = 'f1')
    print(f"{name}: {scores.mean():.4f} +/- {scores.std():.4f}")

Linear Regression: 0.8057 +/- 0.0122
Decision Tree: 0.7651 +/- 0.0276
RandomForestClassifier: 0.8534 +/- 0.0136


In [16]:
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([
        ('Binary_Mapper', Binary_Mapper()),
        ('FeatureEngineer', FeatureEngineer()),
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

    train_acc = accuracy_score(y_train, pipe.predict(X_train))
    test_acc = accuracy_score(y_test, pipe.predict(X_test))
    print(f"{name}: Test: {test_acc:.4f} | Train:  {train_acc:.4f}")

Linear Regression: Test: 0.8537 | Train:  0.7515
Decision Tree: Test: 0.6585 | Train:  1.0000
RandomForestClassifier: Test: 0.8374 | Train:  1.0000


In [ ]:
row = pd.DataFrame([{
    "Gender": "Male", "Married": "no", "Dependents": "0", "Education": "Graduate",
    "Self_Employed": "No", "ApplicantIncome": 5000, "CoapplicantIncome": 1500,
    "LoanAmount": 128, "Loan_Amount_Term": 360, "Credit_History": 1.0,
    "Property_Area": "Urban",
}])
fitted_pipelines['LogisticRegression'].predict_proba(row)

array([[0.19, 0.81]])